# CLARYON Quickstart

End-to-end demo: load Iris data, train XGBoost + quantum kernel SVM, evaluate, and report.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

# Create demo CSV from Iris (binary: setosa vs non-setosa)
iris = load_iris()
df = pd.DataFrame(iris.data, columns=[f"f{i}" for i in range(4)])
df["label"] = (iris.target > 0).astype(int)
df.insert(0, "Key", [f"S{i:04d}" for i in range(len(df))])

tmp = tempfile.mkdtemp()
csv_path = Path(tmp) / "iris_binary.csv"
df.to_csv(csv_path, sep=";", index=False)
print(f"Data saved to {csv_path}")
print(df.head())

In [ ]:
import yaml

config_dict = {
    "experiment": {"name": "quickstart", "seed": 42, "results_dir": str(Path(tmp) / "Results")},
    "data": {"tabular": {"path": str(csv_path), "label_col": "label", "id_col": "Key", "sep": ";"}},
    "cv": {"strategy": "kfold", "n_folds": 3, "seeds": [42]},
    "models": [
        {"name": "mlp", "type": "tabular"},
    ],
    "evaluation": {"metrics": ["bacc", "auc", "sensitivity", "specificity"]},
    "reporting": {"markdown": True, "latex": False},
}

config_path = Path(tmp) / "quickstart.yaml"
with open(config_path, "w") as f:
    yaml.dump(config_dict, f)
print(f"Config saved to {config_path}")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from claryon.config_schema import load_config
from claryon.pipeline import run_pipeline

config = load_config(config_path)
state = run_pipeline(config)

print("\nMetrics summary:")
for model, metrics in state.metrics_summary.items():
    print(f"  {model}: {metrics}")

In [ ]:
# Read the generated report
report_path = Path(tmp) / "Results" / "report.md"
if report_path.exists():
    print(report_path.read_text())
else:
    print("Report not generated (check logs above)")